In [6]:
# imports
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import colors

from fitness_landscape_graph.preprocess import preprocess_data
from fitness_landscape_graph.pair_table_global import calculate_normalized_fitness

In [7]:
base_path = '/endosome/work/greencenter/s439821/fitness-landscape-graph'

wildtype = "............."
dead_mutant = "XXXXXXXXXXXXX"

# File paths
amp_path = f"{base_path}/data/raw/combined-auc/genotype_auc_sorted_ampicillin.csv"
azt_path = f"{base_path}/data/raw/combined-auc/genotype_auc_sorted_aztreonam.csv"
amp_pairs_path = f"{base_path}/data/processed/amp_pairs.csv"
azt_pairs_path = f"{base_path}/data/processed/azt_pairs.csv"

# Load and preprocess data
processed_data = preprocess_data(amp_path, azt_path, amp_pairs_path, azt_pairs_path, clean_nulls_flag=True)

# Access the processed dataframes
amp_df = processed_data['amp']['original']
amp_long_df = processed_data['amp']['long']
amp_pairs_df = processed_data['amp']['pairs']

azt_df = processed_data['azt']['original']
azt_long_df = processed_data['azt']['long']
azt_pairs_df = processed_data['azt']['pairs']


In [8]:
def print_distribution_statistics(values, label=""):
    values = np.abs(values)
    # Calculate percentiles
    p50 = np.percentile(values, 50)
    p75 = np.percentile(values, 75)
    p90 = np.percentile(values, 90)
    p95 = np.percentile(values, 95)
    p99 = np.percentile(values, 99)
    p99_5 = np.percentile(values, 99.5)

    # Calculate other stats
    min_val = values.min()
    max_val = values.max()
    mean_val = values.mean()
    
    # Print the distribution statistics
    print(f"Distribution of {label}:")
    print(f"50th percentile (median): {p50}")
    print(f"75th percentile: {p75}")
    print(f"90th percentile: {p90}")
    print(f"95th percentile: {p95}")
    print(f"99th percentile: {p99}")
    print(f"99.5th percentile: {p99_5}")
    print(f"Min: {min_val}")
    print(f"Max: {max_val}")
    print(f"Mean: {mean_val}")

In [9]:
values = amp_pairs_df.filter(pl.col('concentration') == 0)['median_diff'].to_numpy()
print_distribution_statistics(values, label="amp_fitness_diff")
values = azt_pairs_df.filter(pl.col('concentration') == 0)['median_diff'].to_numpy()
print_distribution_statistics(values, label="azt_fitness_diff")


Distribution of amp_fitness_diff:
50th percentile (median): 0.08152133989988641
75th percentile: 0.143301848590367
90th percentile: 0.21436213481241126
95th percentile: 0.26519559143645954
99th percentile: 0.3846282736475437
99.5th percentile: 0.44050604959593664
Min: 3.0935835670931056e-08
Max: 1.25657261272231
Mean: 0.10191900280651815
Distribution of azt_fitness_diff:
50th percentile (median): 0.08568808422312224
75th percentile: 0.14971289171168634
90th percentile: 0.2235219897087212
95th percentile: 0.27646196438888754
99th percentile: 0.40018273803452453
99.5th percentile: 0.4551867865368862
Min: 2.766330915449089e-07
Max: 0.9820376241379192
Mean: 0.1065671745652155


Based on this distribution, we pick a set of neutral thresholds to test:

0.08, 0.14, 0.22, 0.27, 0.40, 0.45